In [112]:
df_analysis = pd.read_csv("bottleneck_analysis_report.csv")

print("Shape:", df_analysis.shape)

print("\nColumns:")
print(df_analysis.columns.tolist())

print("\nMissing values:")
print(df_analysis.isnull().sum())

print("\nDuplicates:")
print(df_analysis.duplicated().sum())

print("\nFirst rows:")
print(df_analysis.head())

Shape: (235, 5)

Columns:
['line_id', 'bottleneck_station_id', 'count', 'avg_utilisation', 'avg_oee']

Missing values:
line_id                  0
bottleneck_station_id    0
count                    0
avg_utilisation          0
avg_oee                  0
dtype: int64

Duplicates:
0

First rows:
               line_id bottleneck_station_id  count  avg_utilisation   avg_oee
0  MFG-LINE-PLT-003-02                WC-001     80        89.884875  0.627927
1  MFG-LINE-PLT-004-04                WC-002     48        94.044167  0.686223
2  MFG-LINE-PLT-004-02                WC-002     47        96.315957  0.712551
3  MFG-LINE-PLT-004-04                WC-001     43        96.063953  0.666305
4  MFG-LINE-PLT-007-02                WC-001     43        95.167907  0.638735


In [113]:
print(df_analysis.dtypes)

line_id                   object
bottleneck_station_id     object
count                      int64
avg_utilisation          float64
avg_oee                  float64
dtype: object


In [114]:
print("Shape:", df_analysis.shape)

print("\nMissing values:")
print(df_analysis.isnull().sum())

print("\nDuplicates:")
print(df_analysis.duplicated().sum())

print("\nNumber of lines:",
      df_analysis["line_id"].nunique())

print("\nNumber of bottleneck stations:",
      df_analysis["bottleneck_station_id"].nunique())

Shape: (235, 5)

Missing values:
line_id                  0
bottleneck_station_id    0
count                    0
avg_utilisation          0
avg_oee                  0
dtype: int64

Duplicates:
0

Number of lines: 36

Number of bottleneck stations: 20


In [115]:
print(
    df_analysis.sort_values(
        "avg_utilisation",
        ascending=False
    ).head(20)
)

                 line_id bottleneck_station_id  count  avg_utilisation  \
219  MFG-LINE-PLT-006-02                WC-020      3        99.900000   
228  MFG-LINE-PLT-006-02                WC-008      3        99.900000   
223  MFG-LINE-PLT-006-02                WC-003      3        99.900000   
234  MFG-LINE-PLT-006-02                WC-001      1        99.900000   
233  MFG-LINE-PLT-006-01                WC-009      1        99.900000   
232  MFG-LINE-PLT-006-02                WC-013      1        99.900000   
230  MFG-LINE-PLT-006-02                WC-005      2        99.900000   
218  MFG-LINE-PLT-009-01                WC-004      4        99.900000   
177  MFG-LINE-PLT-006-01                WC-005      6        99.690000   
224  MFG-LINE-PLT-006-02                WC-010      3        99.673333   
135  MFG-LINE-PLT-009-01                WC-006      8        99.667500   
205  MFG-LINE-PLT-006-02                WC-006      5        99.620000   
215  MFG-LINE-PLT-006-02              

In [116]:
line_summary = (
    df_analysis
    .groupby("line_id")
    .agg(
        avg_utilisation=("avg_utilisation", "mean"),
        avg_oee=("avg_oee", "mean"),
        bottleneck_count=("count", "sum")
    )
    .reset_index()
)

In [118]:
from sklearn.preprocessing import MinMaxScaler

In [119]:
line_summary = (
    df_analysis
    .groupby("line_id")
    .agg(
        avg_utilisation=("avg_utilisation", "mean"),
        avg_oee=("avg_oee", "mean"),
        bottleneck_count=("count", "sum")
    )
    .reset_index()
)

util_scaler = MinMaxScaler()
oee_scaler = MinMaxScaler()

util_score = util_scaler.fit_transform(
    line_summary[["avg_utilisation"]]
).ravel()

oee_score = oee_scaler.fit_transform(
    line_summary[["avg_oee"]]
).ravel()

line_summary["line_severity"] = (
    0.6 * util_score +
    0.4 * (1 - oee_score)
)

line_summary = line_summary.sort_values(
    "line_severity",
    ascending=False
)

print(line_summary.head(10))

                line_id  avg_utilisation   avg_oee  bottleneck_count  \
24  MFG-LINE-PLT-006-01        99.036405  0.628121                77   
25  MFG-LINE-PLT-006-02        99.481724  0.641421                79   
34  MFG-LINE-PLT-010-03        98.330857  0.618686               101   
30  MFG-LINE-PLT-009-01        98.679773  0.631162                69   
5   MFG-LINE-PLT-002-04        98.696104  0.662482               100   
23  MFG-LINE-PLT-005-06        94.377564  0.577856                89   
6   MFG-LINE-PLT-003-01        94.753571  0.591517                76   
33  MFG-LINE-PLT-010-02        97.850094  0.671784                78   
18  MFG-LINE-PLT-005-01        95.800399  0.637699                74   
1   MFG-LINE-PLT-001-02        97.584215  0.684540                70   

    line_severity  
24       0.843965  
25       0.837885  
34       0.823915  
30       0.813913  
5        0.735053  
23       0.680885  
6        0.669555  
33       0.658437  
18       0.617219  
1      

In [120]:
df_oee = pd.read_csv("oee_summary_by_line.csv")

print("Shape:", df_oee.shape)

print("\nColumns:")
print(df_oee.columns.tolist())

print("\nMissing values:")
print(df_oee.isnull().sum())

print("\nDuplicates:")
print(df_oee.duplicated().sum())

print("\nFirst rows:")
print(df_oee.head())

Shape: (36, 9)

Columns:
['line_id', 'product_family', 'automation_level', 'oee_mean', 'oee_std', 'availability_mean', 'performance_mean', 'quality_mean', 'shifts']

Missing values:
line_id              0
product_family       0
automation_level     0
oee_mean             0
oee_std              0
availability_mean    0
performance_mean     0
quality_mean         0
shifts               0
dtype: int64

Duplicates:
0

First rows:
               line_id   product_family  automation_level  oee_mean   oee_std  \
0  MFG-LINE-PLT-001-01   pharmaceutical        lights_out  0.663071  0.135257   
1  MFG-LINE-PLT-001-02   pharmaceutical            manual  0.686371  0.116085   
2  MFG-LINE-PLT-002-01  electronics_pcb  highly_automated  0.609906  0.145538   
3  MFG-LINE-PLT-002-02  electronics_pcb        lights_out  0.735492  0.118093   
4  MFG-LINE-PLT-002-03  electronics_pcb  highly_automated  0.671665  0.146739   

   availability_mean  performance_mean  quality_mean  shifts  
0           0.879489

In [121]:
line_diagnosis = line_summary.merge(
    df_oee,
    on="line_id",
    how="left"
)

print("Shape:", line_diagnosis.shape)

print(line_diagnosis.head())

Shape: (36, 13)
               line_id  avg_utilisation   avg_oee  bottleneck_count  \
0  MFG-LINE-PLT-006-01        99.036405  0.628121                77   
1  MFG-LINE-PLT-006-02        99.481724  0.641421                79   
2  MFG-LINE-PLT-010-03        98.330857  0.618686               101   
3  MFG-LINE-PLT-009-01        98.679773  0.631162                69   
4  MFG-LINE-PLT-002-04        98.696104  0.662482               100   

   line_severity        product_family automation_level  oee_mean   oee_std  \
0       0.843965         food_beverage   semi_automated  0.639856  0.154337   
1       0.837885         food_beverage   semi_automated  0.652880  0.138206   
2       0.823915  industrial_machinery   semi_automated  0.615436  0.146353   
3       0.813913       automotive_body   semi_automated  0.624228  0.122359   
4       0.735053       electronics_pcb           manual  0.663345  0.150431   

   availability_mean  performance_mean  quality_mean  shifts  
0           0.81022

In [ ]:
print(
    line_diagnosis[
        [
            "line_id",
            "avg_utilisation",
            "avg_oee",
            "bottleneck_count",
            "line_severity",
            "oee_mean",
            "availability_mean",
            "performance_mean",
            "quality_mean",
            "shifts"
        ]
    ].sort_values(
        "line_severity",
        ascending=False
    ).head(10)
)}

In [122]:
line_diagnosis["oee_weakest_factor"] = (
    line_diagnosis[
        [
            "availability_mean",
            "performance_mean",
            "quality_mean"
        ]
    ].idxmin(axis=1)
)

In [123]:
line_diagnosis["oee_weakest_factor"] = (
    line_diagnosis["oee_weakest_factor"]
    .str.replace("_mean", "")
)

In [124]:
print(
    line_diagnosis[
        [
            "line_id",
            "line_severity",
            "oee_mean",
            "availability_mean",
            "performance_mean",
            "quality_mean",
            "oee_weakest_factor"
        ]
    ].sort_values(
        "line_severity",
        ascending=False
    ).head(15)
)

                line_id  line_severity  oee_mean  availability_mean  \
0   MFG-LINE-PLT-006-01       0.843965  0.639856           0.810226   
1   MFG-LINE-PLT-006-02       0.837885  0.652880           0.839777   
2   MFG-LINE-PLT-010-03       0.823915  0.615436           0.835797   
3   MFG-LINE-PLT-009-01       0.813913  0.624228           0.830981   
4   MFG-LINE-PLT-002-04       0.735053  0.663345           0.865270   
5   MFG-LINE-PLT-005-06       0.680885  0.577966           0.779374   
6   MFG-LINE-PLT-003-01       0.669555  0.597887           0.792525   
7   MFG-LINE-PLT-010-02       0.658437  0.672685           0.885883   
8   MFG-LINE-PLT-005-01       0.617219  0.639681           0.840868   
9   MFG-LINE-PLT-001-02       0.609281  0.686371           0.902366   
10  MFG-LINE-PLT-004-03       0.606647  0.585064           0.774715   
11  MFG-LINE-PLT-004-01       0.605311  0.609943           0.783780   
12  MFG-LINE-PLT-005-04       0.595386  0.636909           0.838342   
13  MF

In [125]:
df_pareto = pd.read_csv("downtime_pareto.csv")

print("Shape:", df_pareto.shape)

print("\nColumns:")
print(df_pareto.columns.tolist())

print("\nMissing values:")
print(df_pareto.isnull().sum())

print("\nDuplicates:")
print(df_pareto.duplicated().sum())

print("\nFirst rows:")
print(df_pareto.head())


Shape: (6, 3)

Columns:
['category', 'total_minutes', 'cumulative_pct']

Missing values:
category          0
total_minutes     0
cumulative_pct    0
dtype: int64

Duplicates:
0

First rows:
                         category  total_minutes  cumulative_pct
0    downtime_category_mechanical        91668.1       49.483001
1    downtime_category_electrical        35032.7       68.393866
2       downtime_category_tooling        23179.5       80.906302
3  downtime_category_quality_hold        14965.9       88.984986
4      downtime_category_material        13654.2       96.355607
